In [ ]:
# The libraries 
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Filesystems to compare
filesystems = ["ext4", "xfs", "btrfs", "ntfs", "vfat"]

# Store all benchmark rows
dfs = []

for fs in filesystems:
  path = f"./{fs}/out/ondemand/*.log"

  for file in glob.glob(path):

    # Extract benchmark size from filename
    size = int(os.path.basename(file).replace(".log", ""))

    # Load CSV-like bonnie++ output
    df = pd.read_csv(file)

    # Add metadata columns
    df["filesystem"] = fs
    df["size"] = size
    
    dfs.append(df)

# Combine everything
data = pd.concat(dfs, ignore_index=True)

# Clean data
# Remove "+++++" overflow values
data = data[data["put_block"] != "+++++"]

# Convert throughput to numeric
data["put_block"] = pd.to_numeric(data["put_block"])

# Remove missing latency values
data = data.dropna(subset=["put_block_latency"])

data.head()
data.columns

In [ ]:
# -----------------------------
# FILE I/O TESTS
# -----------------------------

# Metrics to compare
io_tests = [
    ("put_block", "put_block_latency", "Block Write"),
    ("get_block", "get_block_latency", "Block Read"),
    ("rewrite", "rewrite_latency", "Rewrite"),
]

for throughput_col, latency_col, title in io_tests:

    # Remove invalid rows
    plot_data = data.dropna(subset=[throughput_col, latency_col])

    # Remove bonnie++ overflow values
    plot_data = plot_data[plot_data[throughput_col] != "+++++"]

    # Convert to numeric
    plot_data[throughput_col] = pd.to_numeric(
        plot_data[throughput_col],
        errors="coerce"
    )

    plt.figure(figsize=(8,6))

    sns.scatterplot(
        data=plot_data,
        x=throughput_col,
        y=latency_col,
        hue="filesystem",
        style="filesystem",
        s=100
    )

    plt.title(f"{title}: Throughput vs Latency")
    plt.xlabel("Throughput")
    plt.ylabel("Latency (ms)")
    plt.grid(True)

    plt.show()

In [ ]:
# -----------------------------
# FILE CREATION TESTS
# -----------------------------

creation_tests = [
    ("seq_create", "seq_create_latency", "Sequential File Creation"),
    ("ran_create", "ran_create_latency", "Random File Creation"),
]

for throughput_col, latency_col, title in creation_tests:

    plot_data = data.dropna(subset=[throughput_col, latency_col])

    plot_data[throughput_col] = pd.to_numeric(
        plot_data[throughput_col],
        errors="coerce"
    )

    plt.figure(figsize=(8,6))

    sns.scatterplot(
        data=plot_data,
        x=throughput_col,
        y=latency_col,
        hue="filesystem",
        style="filesystem",
        s=100
    )

    plt.title(f"{title}: Throughput vs Latency")
    plt.xlabel("Operations/sec")
    plt.ylabel("Latency (ms)")
    plt.grid(True)

    plt.show()

Logs to choose
- 64
- 256
- 1024

In [ ]:
# Importing the logs of interest
# the path to the logs just might need to be changed according to each one. 
# I'm connecting my vscode to github so that's the path for the files.
# It might be different for someone else

btrfs64 = pd.read_table("./btrfs/out/ondemand/64.log", sep=",")
btrfs256 = pd.read_table("./btrfs/out/ondemand/256.log", sep=",")
btrfs1024 = pd.read_table("./btrfs/out/ondemand/1024.log", sep=",")

ext4_64= pd.read_table("./ext4/out/ondemand/64.log", sep=",")
ext4_256 = pd.read_table("./ext4/out/ondemand/256.log", sep=",")
ext4_1024 = pd.read_table("./ext4/out/ondemand/1024.log", sep=",")

ntfs64 = pd.read_table("./ntfs/out/ondemand/64.log", sep=",")
ntfs256 = pd.read_table("./ntfs/out/ondemand/256.log", sep=",")
ntfs1024 = pd.read_table("./ntfs/out/ondemand/1024.log", sep=",")

vfat64 = pd.read_table("./vfat/out/ondemand/64.log", sep=",")
vfat256 = pd.read_table("./vfat/out/ondemand/256.log", sep=",")
vfat1024 = pd.read_table("./vfat/out/ondemand/1024.log", sep=",")

xfs64 = pd.read_table("./xfs/out/ondemand/64.log", sep=",")
xfs256 = pd.read_table("./xfs/out/ondemand/256.log", sep=",")
xfs1024 = pd.read_table("./xfs/out/ondemand/1024.log", sep=",")

In [ ]:
btrfs64[""]

In [ ]:
# Useful for the plotting
btrfs = [
    (btrfs64,   "btrfs-64",   "#000080", "o"),
    (btrfs256,  "btrfs-256",  "#D2042D", "s"),
    (btrfs1024, "btrfs-1024", "#228B22", "^"),
]

ext4 = [
    (ext4_64,   "ext4-64",   "#000080", "o"),
    (ext4_256,  "ext4-256",  "#D2042D", "s"),
    (ext4_1024, "ext4-1024", "#228B22", "^"),
]

ntfs = [
    (ntfs64,   "ntfs-64",   "#000080", "o"),
    (ntfs256,  "ntfs-256",  "#D2042D", "s"),
    (ntfs1024, "ntfs-1024", "#228B22", "^"),
]

vfat = [
    (vfat64,   "vfat-64",   "#000080", "o"),
    (vfat256,  "vfat-256",  "#D2042D", "s"),
    (vfat1024, "vfat-1024", "#228B22", "^"),
]

xfs = [
    (xfs64,   "xfs-64",   "#000080", "o"),
    (xfs256,  "xfs-256",  "#D2042D", "s"),
    (xfs1024, "xfs-1024", "#228B22", "^"),
]

### Doing the plots for "get_block" and "get_block_latency", all filesystems

In [ ]:
# btrfs Scatter plot

fig, ax = plt.subplots(figsize=(10, 6))

for df, label, color, marker in btrfs:
    ax.scatter(
        df["get_block"],
        df["get_block_latency"],
        label=label,
        color=color,
        marker=marker,
        alpha=0.6,
        s=40,
        edgecolors="none",
    )

ax.set_xlabel("get_block", fontsize=12)
ax.set_ylabel("get_block_latency", fontsize=12)
ax.set_title("get_block vs get_block_latency (btrfs ondemand)", fontsize=14)
ax.legend(title="Dataset", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("btrfs_scatter_getblock.png", dpi=150)
plt.show()
print("Saved to btrfs_scatter_getblock.png")

In [ ]:
# ext4 Scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

for df, label, color, marker in ext4:
    ax.scatter(
        df["get_block"],
        df["get_block_latency"],
        label=label,
        color=color,
        marker=marker,
        alpha=0.6,
        s=40,
        edgecolors="none",
    )

ax.set_xlabel("get_block", fontsize=12)
ax.set_ylabel("get_block_latency", fontsize=12)
ax.set_title("get_block vs get_block_latency (ext4 ondemand)", fontsize=14)
ax.legend(title="Dataset", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("ext4_scatter_getblock.png", dpi=150)
plt.show()
print("Saved to ext4_scatter_getblock.png")

In [ ]:
# ntfs scatter plot

fig, ax = plt.subplots(figsize=(10, 6))

for df, label, color, marker in ntfs:
    ax.scatter(
        df["get_block"],
        df["get_block_latency"],
        label=label,
        color=color,
        marker=marker,
        alpha=0.6,
        s=40,
        edgecolors="none",
    )

ax.set_xlabel("get_block", fontsize=12)
ax.set_ylabel("get_block_latency", fontsize=12)
ax.set_title("get_block vs get_block_latency (ntfs ondemand)", fontsize=14)
ax.legend(title="Dataset", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("ntfs_scatter_getblock.png", dpi=150)
plt.show()
print("Saved to ntfs_scatter_getblock.png")

In [ ]:
# vfat scatter plot

fig, ax = plt.subplots(figsize=(10, 6))

for df, label, color, marker in vfat:
    ax.scatter(
        df["get_block"],
        df["get_block_latency"],
        label=label,
        color=color,
        marker=marker,
        alpha=0.6,
        s=40,
        edgecolors="none",
    )

ax.set_xlabel("get_block", fontsize=12)
ax.set_ylabel("get_block_latency", fontsize=12)
ax.set_title("get_block vs get_block_latency (vfat ondemand)", fontsize=14)
ax.legend(title="Dataset", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("vfat_scatter_getblock.png", dpi=150)
plt.show()
print("Saved to vfat_scatter_getblock.png")

In [ ]:
# xfs scatter plot

fig, ax = plt.subplots(figsize=(10, 6))

for df, label, color, marker in xfs:
    ax.scatter(
        df["get_block"],
        df["get_block_latency"],
        label=label,
        color=color,
        marker=marker,
        alpha=0.6,
        s=40,
        edgecolors="none",
    )

ax.set_xlabel("get_block", fontsize=12)
ax.set_ylabel("get_block_latency", fontsize=12)
ax.set_title("get_block vs get_block_latency (xfs ondemand)", fontsize=14)
ax.legend(title="Dataset", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("xfs_scatter_getblock.png", dpi=150)
plt.show()
print("Saved to xfs_scatter_getblock.png")